# Inventory Planning Agent v2a — DB-backed RAG

Extends v2 with an **in-memory SQLite database** so the user only needs to provide a **SKU**:
- Inventory, lead-time, and demand data are stored in three relational tables
- The agent looks up all parameters by SKU, then runs the same RAG-enhanced chains from v2
- ChromaDB + Sentence-Transformers still provide domain knowledge retrieval

In [ ]:
# sqlite3 is built into Python — no extra install needed
!pip install langchain langchain-groq langchain-chroma langchain-huggingface chromadb sentence-transformers gradio -q

In [ ]:
import getpass, os
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

## In-Memory Relational Database

Three tables store SKU master data. `sqlite3` is part of the Python standard library — no extra package required.

In [ ]:
import sqlite3

# ':memory:' keeps the DB in RAM — data is lost when the runtime restarts
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row   # rows accessible by column name
cur = conn.cursor()

# ── Schema ────────────────────────────────────────────────────────────────
cur.executescript("""
CREATE TABLE inventory (
    sku_id        TEXT PRIMARY KEY,
    sku_name      TEXT NOT NULL,
    category      TEXT,
    current_stock REAL NOT NULL,
    service_level REAL NOT NULL   -- target % e.g. 95
);

CREATE TABLE lead_time (
    sku_id         TEXT PRIMARY KEY,
    lead_time_days REAL NOT NULL,
    supplier_name  TEXT,
    FOREIGN KEY (sku_id) REFERENCES inventory(sku_id)
);

CREATE TABLE demand (
    sku_id           TEXT PRIMARY KEY,
    avg_daily_demand REAL NOT NULL,
    demand_std_dev   REAL,          -- optional: for advanced safety-stock calc
    FOREIGN KEY (sku_id) REFERENCES inventory(sku_id)
);
""")

# ── Sample data ───────────────────────────────────────────────────────────
inventory_rows = [
    ("SKU-001", "Wireless Headphones",    "Electronics",    200,  98),
    ("SKU-002", "Paracetamol 500mg",      "Pharma",         800,  99),
    ("SKU-003", "Instant Noodles 12-pack","FMCG",          1200,  95),
    ("SKU-004", "Steel Rod 10mm",         "Raw Materials",   300,  92),
    ("SKU-005", "Laptop 15-inch",         "Electronics",     50,  99),
    ("SKU-006", "Vitamin C Tablets",      "Pharma",         600,  97),
    ("SKU-007", "Shampoo 400ml",          "FMCG",           900,  95),
    ("SKU-008", "Copper Wire 2.5mm",      "Raw Materials",   150,  90),
    ("SKU-009", "Bluetooth Speaker",      "Electronics",    120,  96),
    ("SKU-010", "Flour 5kg",              "FMCG",          2000,  95),
]

lead_time_rows = [
    ("SKU-001", 21, "TechImports Ltd"),
    ("SKU-002",  7, "MedSupply Co"),
    ("SKU-003",  4, "FoodDist Pvt"),
    ("SKU-004", 14, "SteelWorks Inc"),
    ("SKU-005", 30, "GlobalTech"),
    ("SKU-006", 10, "PharmaPlus"),
    ("SKU-007",  5, "FMCG Direct"),
    ("SKU-008", 21, "CopperMine Co"),
    ("SKU-009", 18, "AudioWorld"),
    ("SKU-010",  3, "LocalMills"),
]

demand_rows = [
    ("SKU-001",  8,  2.0),
    ("SKU-002", 40,  5.0),
    ("SKU-003", 80, 10.0),
    ("SKU-004", 15,  3.0),
    ("SKU-005",  3,  1.0),
    ("SKU-006", 25,  4.0),
    ("SKU-007", 60,  8.0),
    ("SKU-008", 10,  2.5),
    ("SKU-009",  6,  1.5),
    ("SKU-010",150, 20.0),
]

cur.executemany("INSERT INTO inventory VALUES (?,?,?,?,?)", inventory_rows)
cur.executemany("INSERT INTO lead_time VALUES (?,?,?)",     lead_time_rows)
cur.executemany("INSERT INTO demand   VALUES (?,?,?)",      demand_rows)
conn.commit()

# ── Helper: fetch SKU record ───────────────────────────────────────────────
def fetch_sku(sku_id: str) -> dict | None:
    """Return a dict with all inventory parameters for a SKU, or None if not found."""
    row = cur.execute("""
        SELECT i.sku_id, i.sku_name, i.category,
               i.current_stock, i.service_level,
               l.lead_time_days, l.supplier_name,
               d.avg_daily_demand, d.demand_std_dev
        FROM   inventory i
        JOIN   lead_time l ON l.sku_id = i.sku_id
        JOIN   demand    d ON d.sku_id = i.sku_id
        WHERE  UPPER(i.sku_id) = UPPER(?)
    """, (sku_id,)).fetchone()
    return dict(row) if row else None


def list_skus() -> list[dict]:
    """Return all SKUs in the database."""
    rows = cur.execute(
        "SELECT sku_id, sku_name, category, current_stock FROM inventory ORDER BY sku_id"
    ).fetchall()
    return [dict(r) for r in rows]


# ── Sanity check ──────────────────────────────────────────────────────────
print(f"DB initialised with {len(inventory_rows)} SKUs\n")
print(f"{'SKU':<10} {'Name':<28} {'Category':<16} {'Stock':>6}")
print("-" * 64)
for s in list_skus():
    print(f"{s['sku_id']:<10} {s['sku_name']:<28} {s['category']:<16} {s['current_stock']:>6}")

## RAG Knowledge Base

Same domain-knowledge documents as v2, stored in ChromaDB.

In [ ]:
KB_DOCUMENTS = [
    {
        "id": "kb-formula-safety-stock",
        "text": (
            "Safety Stock Formula: Safety Stock = Z * sqrt(Lead Time) * Average Daily Demand. "
            "Z-scores by service level: 90% -> 1.28, 95% -> 1.65, 98% -> 2.05, 99% -> 2.33."
        ),
        "metadata": {"topic": "formula", "subtopic": "safety_stock"}
    },
    {
        "id": "kb-formula-reorder-point",
        "text": "Reorder Point (ROP) = (Average Daily Demand * Lead Time) + Safety Stock.",
        "metadata": {"topic": "formula", "subtopic": "reorder_point"}
    },
    {
        "id": "kb-formula-desired-stock",
        "text": (
            "Desired Stock Level = (Average Daily Demand * Lead Time) + Safety Stock. "
            "Reorder Quantity = max(0, Desired Stock - Current Stock)."
        ),
        "metadata": {"topic": "formula", "subtopic": "desired_stock"}
    },
    {
        "id": "kb-formula-eoq",
        "text": "Economic Order Quantity (EOQ) = sqrt((2 * Annual Demand * Ordering Cost) / Holding Cost per Unit).",
        "metadata": {"topic": "formula", "subtopic": "eoq"}
    },
    {
        "id": "kb-category-electronics",
        "text": "Electronics: lead times 14-30 days, service level 95-99%, high obsolescence risk.",
        "metadata": {"topic": "category", "subtopic": "electronics"}
    },
    {
        "id": "kb-category-fmcg",
        "text": "FMCG: lead times 3-7 days, high steady demand, service level 95-98%.",
        "metadata": {"topic": "category", "subtopic": "fmcg"}
    },
    {
        "id": "kb-category-pharma",
        "text": "Pharma: lead times 7-21 days, service level 99%+, strict FIFO and expiry management.",
        "metadata": {"topic": "category", "subtopic": "pharma"}
    },
    {
        "id": "kb-category-raw-materials",
        "text": "Raw materials: lead times 10-45 days, service level 90-95%.",
        "metadata": {"topic": "category", "subtopic": "raw_materials"}
    },
    {
        "id": "kb-bp-abc-analysis",
        "text": "ABC Analysis: A-items need tight control and high service levels; C-items tolerate bulk ordering.",
        "metadata": {"topic": "best_practice", "subtopic": "abc_analysis"}
    },
    {
        "id": "kb-bp-demand-variability",
        "text": "High demand variability requires more safety stock to maintain the same service level.",
        "metadata": {"topic": "best_practice", "subtopic": "demand_variability"}
    },
    {
        "id": "kb-bp-seasonal",
        "text": "Increase safety stock 4-6 weeks before peak season; reduce quickly post-season.",
        "metadata": {"topic": "best_practice", "subtopic": "seasonal"}
    },
    {
        "id": "kb-bp-supplier-risk",
        "text": "Single-source or long international lead times: add 10-20% risk buffer to safety stock.",
        "metadata": {"topic": "best_practice", "subtopic": "supplier_risk"}
    },
    {
        "id": "kb-bp-service-level-tradeoff",
        "text": "Raising service level from 95% to 99% roughly doubles safety stock (Z: 1.65 -> 2.33).",
        "metadata": {"topic": "best_practice", "subtopic": "service_level"}
    },
]

print(f"Knowledge base: {len(KB_DOCUMENTS)} documents loaded")

## ChromaDB Setup & Embedding Model Instantiation

In [ ]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

chroma_client = chromadb.Client()
COLLECTION_NAME = "inventory_knowledge"
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print(f"Embedding model    : {EMBED_MODEL_NAME}")
print(f"ChromaDB collection: '{COLLECTION_NAME}' (in-memory)")

## Embedding Generation & ChromaDB Population

In [ ]:
texts     = [doc["text"]     for doc in KB_DOCUMENTS]
ids       = [doc["id"]       for doc in KB_DOCUMENTS]
metadatas = [doc["metadata"] for doc in KB_DOCUMENTS]

print("Generating embeddings...")
vectors = embeddings.embed_documents(texts)
collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metadatas)
print(f"Stored {collection.count()} documents in ChromaDB")


def retrieve_context(query: str, n_results: int = 3) -> str:
    """Return top-k knowledge chunks most relevant to query."""
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=n_results)
    return "\n\n".join(f"- {c}" for c in results["documents"][0])


# Smoke-test
sample = retrieve_context("safety stock formula")
print("\nSample retrieval:")
print(sample[:200], "...")

## LLM & Prompt Chains

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [ ]:
# Chain 1: Compute desired stock (same as v2, RAG context injected)
compute_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an inventory planning expert. Given inventory parameters, compute the desired stock level.

Use this formula:
- Safety Stock = Z-score(service_level) * sqrt(lead_time) * average_daily_demand
- Desired Stock = (average_daily_demand * lead_time) + Safety Stock

Common Z-scores: 90% -> 1.28, 95% -> 1.65, 98% -> 2.05, 99% -> 2.33

Additional domain knowledge:
{rag_context}

Show your step-by-step calculation. End your response with a line:
DESIRED_STOCK: <number>"""),
    ("human", "SKU: {sku_id} | {sku_name} ({category})\nCurrent Stock: {current_stock}\nLead Time (days): {lead_time_days}\nAverage Daily Demand: {avg_daily_demand}\nService Level: {service_level}%")
])
compute_chain = compute_prompt | llm | StrOutputParser()

# Chain 2: Reorder recommendation (same as v2, RAG context injected)
reorder_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an inventory planning advisor. Given the stock computation and the current stock,
determine the reorder quantity and provide a clear recommendation.

Reorder Quantity = max(0, Desired Stock - Current Stock)

Additional domain knowledge:
{rag_context}

Provide:
1. The reorder quantity
2. Whether an immediate reorder is needed
3. A brief justification referencing best practices where relevant"""),
    ("human", "SKU: {sku_id} | {sku_name}\nCurrent Stock: {current_stock}\nSupplier: {supplier_name}\n\nStock Computation:\n{computation}")
])
reorder_chain = reorder_prompt | llm | StrOutputParser()

print("Chains initialised.")

In [ ]:
import re

SKU_PATTERN = re.compile(r"\bSKU-\d{3}\b", re.IGNORECASE)


def extract_sku(user_message: str) -> str | None:
    """Pull the first SKU-XXX token from the user's message."""
    match = SKU_PATTERN.search(user_message)
    return match.group(0).upper() if match else None


def run_inventory_agent(user_message: str) -> str:
    """DB-backed, RAG-enhanced inventory planning agent."""
    # ── Handle 'list' queries ───────────────────────────────────────────────
    if re.search(r"\b(list|show|all|available)\b", user_message, re.IGNORECASE):
        skus = list_skus()
        rows = "\n".join(
            f"- **{s['sku_id']}** — {s['sku_name']} ({s['category']}) | Stock: {s['current_stock']}"
            for s in skus
        )
        return f"**Available SKUs:**\n\n{rows}"

    # ── Extract SKU from message ────────────────────────────────────────────
    sku_id = extract_sku(user_message)
    if not sku_id:
        return (
            "Please include a SKU ID in your message (e.g. **SKU-001**).\n\n"
            "Type **list** to see all available SKUs."
        )

    # ── Fetch parameters from DB ────────────────────────────────────────────
    record = fetch_sku(sku_id)
    if record is None:
        return (
            f"SKU **{sku_id}** was not found in the database.\n\n"
            "Type **list** to see all available SKUs."
        )

    # ── Retrieve RAG context ────────────────────────────────────────────────
    rag_query = f"{record['category']} inventory reorder safety stock service level {record['service_level']}%"
    rag_context = retrieve_context(rag_query)

    # ── Chain 1: Compute desired stock ─────────────────────────────────────
    computation = compute_chain.invoke({
        "sku_id":          record["sku_id"],
        "sku_name":        record["sku_name"],
        "category":        record["category"],
        "current_stock":   record["current_stock"],
        "lead_time_days":  record["lead_time_days"],
        "avg_daily_demand":record["avg_daily_demand"],
        "service_level":   record["service_level"],
        "rag_context":     rag_context,
    })

    # ── Chain 2: Reorder recommendation ────────────────────────────────────
    recommendation = reorder_chain.invoke({
        "sku_id":        record["sku_id"],
        "sku_name":      record["sku_name"],
        "current_stock": record["current_stock"],
        "supplier_name": record["supplier_name"],
        "computation":   computation,
        "rag_context":   rag_context,
    })

    # ── Format response ─────────────────────────────────────────────────────
    db_summary = (
        f"**SKU:** {record['sku_id']} — {record['sku_name']}  \n"
        f"**Category:** {record['category']}  \n"
        f"**Current Stock:** {record['current_stock']} units  \n"
        f"**Lead Time:** {record['lead_time_days']} days (Supplier: {record['supplier_name']})  \n"
        f"**Avg Daily Demand:** {record['avg_daily_demand']} units/day  \n"
        f"**Service Level:** {record['service_level']}%"
    )

    return (
        f"## DB Record\n\n{db_summary}\n\n"
        f"---\n\n**Retrieved Knowledge:**\n{rag_context}\n\n"
        f"---\n\n## Stock Computation\n\n{computation}\n\n"
        f"---\n\n## Reorder Recommendation\n\n{recommendation}"
    )

## Gradio Chat Interface

In [ ]:
import gradio as gr

def chat_handler(message, history):
    try:
        return run_inventory_agent(message)
    except Exception as e:
        return (
            f"Something went wrong: {e}\n\n"
            "Try asking: *\"What is the reorder quantity for SKU-001?\"*  \n"
            "Or type **list** to see all SKUs."
        )

demo = gr.ChatInterface(
    fn=chat_handler,
    title="Inventory Planning Agent v2a (DB + RAG)",
    description=(
        "Just ask by SKU — all inventory, lead-time, and demand data come from the database.  \n"
        "Type **list** to see available SKUs."
    ),
    examples=[
        "list",
        "What is the reorder quantity for SKU-001?",
        "Should I reorder SKU-002?",
        "Give me the reorder recommendation for SKU-005.",
        "How much should I order for SKU-010?",
    ],
    type="messages",
)

demo.launch(debug=True)